<a href="https://colab.research.google.com/github/KDT-GrabIT/train_YOLOX-Nano_by_640x640/blob/feature%2FPP-47%2Ftranfer-fp16/convert_YOLOX_to_TFLite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLOX best_ckpt.pth → Float16 TFLite 변환
구글 드라이브의 best_ckpt.pth를 Float16 TFLite 파일로 변환합니다.

In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ========== 설정 (필요시 수정) ==========
CKPT_PATH = "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/Others/mid/Weights-YOLOX background/yolox_nano_standard/best_ckpt.pth"
OUTPUT_DIR = "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/Others/mid/Weights-YOLOX background"  # TFLite 저장 경로

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"체크포인트: {CKPT_PATH}")
print(f"출력 경로: {OUTPUT_DIR}")
print(f"파일 존재: {os.path.exists(CKPT_PATH)}")

체크포인트: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/Others/mid/Weights-YOLOX background/yolox_nano_standard/best_ckpt.pth
출력 경로: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/Others/mid/Weights-YOLOX background
파일 존재: True


In [ ]:
# 1. YOLOX 클론 및 설치
!git clone https://github.com/Megvii-BaseDetection/YOLOX.git
%cd YOLOX
!pip install -q -r requirements.txt
!pip install -q -v -e . --no-build-isolation
!pip install -q onnx onnxsim onnxruntime onnx2tf tensorflow

Cloning into 'YOLOX'...
remote: Enumerating objects: 1940, done.
remote: Total 1940 (delta 0), reused 0 (delta 0), pack-reused 1940 (from 1)
Receiving objects: 100% (1940/1940), 7.55 MiB | 28.74 MiB/s, done.
Resolving deltas: 100% (1163/1163), done.
/content/YOLOX/YOLOX
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Obtaining file:///content/YOLOX/YOLOX
  Preparing metadata (setup.py) ... done
  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
  Using cached thop-0.1.1.post2209072238-py3-none-any.whl.metadata (2.7 kB)
  Using cached ninja-1.13.0

In [ ]:
# PyTorch 2.6+ 가중치 로딩 호환 패치
import pathlib, re
p = pathlib.Path("tools/export_onnx.py")
if p.exists():
    t = p.read_text()
    if 'weights_only' not in t:
        t = t.replace('torch.load(ckpt_file, map_location="cpu")', 'torch.load(ckpt_file, map_location="cpu", weights_only=False)')
        p.write_text(t)
        print("✅ export_onnx.py 패치 적용")

✅ export_onnx.py 패치 적용


In [ ]:
# 2. 50클래스용 exp 파일 생성
exp_content = '''import os
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()
        self.num_classes = 49
        self.depth = 0.33
        self.width = 0.25
        self.exp_name = "yolox_nano_standard"
        self.test_size = (640, 640)
'''

os.makedirs("exps/custom", exist_ok=True)
with open("exps/custom/yolox_nano_standard.py", "w") as f:
    f.write(exp_content)
print("✅ exps/custom/yolox_nano_standard.py 생성")

✅ exps/custom/yolox_nano_standard.py 생성


In [ ]:
!pip install loguru -q
!pip install onnxscript -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 9.7 MB/s eta 0:00:00


In [ ]:
# export_onnx.py 패치 - 실제 사용 경로 기준
import pathlib

p = pathlib.Path("/content/YOLOX/YOLOX/tools/export_onnx.py")
t = p.read_text()
t = t.replace("torch.onnx._export(", "torch.onnx.export(")
p.write_text(t)

print("✅ 실제 export_onnx.py 패치 완료")


✅ 실제 export_onnx.py 패치 완료


In [ ]:
# 3. PyTorch → ONNX 내보내기
!PYTHONPATH=. python tools/export_onnx.py -f exps/custom/yolox_nano_standard.py -c '{CKPT_PATH}' --output-name yolox_nano_49cls.onnx --batch-size 1 --decode_in_inference

2026-02-09 05:37:19.664 | INFO     | __main__:main:64 - args value: Namespace(output_name='yolox_nano_49cls.onnx', input='images', output='output', opset=11, batch_size=1, dynamic=False, no_onnxsim=False, exp_file='exps/custom/yolox_nano_standard.py', experiment_name=None, name=None, ckpt='/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/Others/mid/Weights-YOLOX background/yolox_nano_standard/best_ckpt.pth', opts=[], decode_in_inference=True)
2026-02-09 05:37:19.937 | INFO     | __main__:main:88 - loading checkpoint done.
W0209 05:37:20.385000 38428 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=1

In [ ]:
# 4. ONNX → TFLite (float32 + float16 자동 생성)
# 입력 shape 고정: 1,3,640,640 (NCHW)
!onnx2tf -i yolox_nano_49cls.onnx -o saved_model -ois images:1,3,640,640 -v info

E0000 00:00:1770615495.836788   38659 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770615495.933186   38659 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770615496.861998   38659 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770615496.862103   38659 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770615496.862109   38659 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770615496.862115   38659 computation_placer.cc:177] computation placer already registered. Please check linka

In [ ]:
# 5. Float16 TFLite를 출력 폴더로 복사
import shutil
import glob

dest = os.path.join(OUTPUT_DIR, "yolox_nano_49cls_float16.tflite")
float16_tflite = glob.glob("saved_model/*_float16.tflite")
if float16_tflite:
    shutil.copy(float16_tflite[0], dest)
    print(f"✅ 저장됨: {dest}")
else:
    all_tflite = glob.glob("saved_model/*.tflite")
    if all_tflite:
        shutil.copy(all_tflite[0], dest)
        print(f"✅ 저장됨: {dest}")
    else:
        print("saved_model 폴더 내용:")
        !ls -la saved_model/

✅ 저장됨: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/Others/mid/Weights-YOLOX background/yolox_nano_49cls_float16.tflite
